# Phase 1 - YOLO11n Training Notebook

## Workflow
このノートブックをColab上で上から順番に最後まで実行してください。

- Ultralyticsをインストールし、GPU環境が正常に認識されていることを確認する
- RoboflowからエクスポートしたYOLOv11用ZIPファイルを手動でアップロードする
- data.yamlを検索し、Colab環境で使用できるようにデータセットのsplitパスを正規化する
- train / validation / test のデータパスとクラス名が正しいか検証する
- Precision重視の設定でYOLO11n（yolo11n.pt）の学習を実行する
- Validationデータを使用してconfidence thresholdのスキャンを行い、デプロイ用の最適な閾値を決定する
- 決定したthresholdを使用してbest.ptをtest splitで評価する
- 学習結果に必要な重要な成果物をZIPアーカイブとしてまとめる
- best.ptと成果物ZIPファイルをダウンロードする
- 必要に応じて成果物をGoogle Driveへバックアップする

In [ ]:
# 手順1: Kaggle 学習環境を確認する

# Cell 1: Kaggle environment check

import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch path:", torch.__file__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nTesting torch._inductor...")

try:
    import torch._inductor
    print("torch._inductor: OK")
except Exception as e:
    print("torch._inductor FAILED:")
    print(type(e).__name__, e)
    raise

In [ ]:
from pathlib import Path

print("Searching for data.yaml...")

data_candidates = list(Path("/kaggle/input").rglob("data.yaml"))

if not data_candidates:
    raise FileNotFoundError("data.yaml not found")

DATA_YAML_PATH = data_candidates[0]

print(f"Found data.yaml: {DATA_YAML_PATH}")

In [ ]:
print(open(DATA_YAML_PATH, "r").read())

In [ ]:
# 手順3: data.yaml を見つけて Kaggle 用に絶対パスで正規化する

from pathlib import Path
import yaml

def find_data_yaml(root_dir):
    yaml_candidates = sorted(root_dir.rglob("data.yaml"))

    if not yaml_candidates:
        raise FileNotFoundError("No data.yaml was found.")

    if len(yaml_candidates) == 1:
        return yaml_candidates[0]

    print("Multiple data.yaml files were found:")
    for i, candidate in enumerate(yaml_candidates, start=1):
        print(f"{i}: {candidate}")

    return yaml_candidates[0]


def choose_existing_split(split_root, candidates, split_name):
    for relative_path in candidates:
        resolved = split_root / relative_path
        if resolved.exists():
            return str(resolved.resolve())

    raise FileNotFoundError(
        f"Could not find valid path for {split_name}: {candidates}"
    )


# Kaggle Input から data.yaml を検索
data_yaml_path = find_data_yaml(Path("/kaggle/input"))

print(f"Located data.yaml: {data_yaml_path}")


# YAML読み込み
with data_yaml_path.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)


# dataset root
split_root = data_yaml_path.parent


# 絶対パス化
train_split = choose_existing_split(
    split_root,
    ["train/images", "train/image"],
    "train"
)

val_split = choose_existing_split(
    split_root,
    ["valid/images", "val/images", "validation/images"],
    "validation"
)

test_split = choose_existing_split(
    split_root,
    ["test/images"],
    "test"
)


data_config["train"] = train_split
data_config["val"] = val_split
data_config["test"] = test_split


# Kaggle Inputは読み取り専用なので working に保存
DATA_YAML_PATH = Path("/kaggle/working/data.yaml")

with DATA_YAML_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        data_config,
        file,
        sort_keys=False,
        allow_unicode=True
    )


print("Normalized data.yaml paths:")
print(f" - train: {data_config['train']}")
print(f" - val:   {data_config['val']}")
print(f" - test:  {data_config['test']}")

print(f"\nTraining YAML saved to: {DATA_YAML_PATH}")

In [ ]:
# 手順4: data.yaml の内容を検証し、データセットの健全性を詳細確認する (Kaggle版)

from collections import Counter
from pathlib import Path
from PIL import Image as PILImage
import yaml


def normalize_class_names(names_field):
    if isinstance(names_field, list):
        return [str(name) for name in names_field]

    if isinstance(names_field, dict):
        try:
            sorted_items = sorted(
                names_field.items(),
                key=lambda item: int(item[0])
            )
        except Exception:
            sorted_items = sorted(
                names_field.items(),
                key=lambda item: str(item[0])
            )

        return [str(name) for _, name in sorted_items]

    raise TypeError(
        "The names field in data.yaml must be a list or dict."
    )


def count_split_contents(images_path_str):
    img_dir = Path(images_path_str)

    # Roboflow YOLO structure:
    # train/images -> train/labels
    label_dir = img_dir.parent / "labels"

    img_exts = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"
    }

    img_count = (
        sum(
            1 for f in img_dir.iterdir()
            if f.suffix.lower() in img_exts
        )
        if img_dir.exists()
        else 0
    )

    lbl_count = (
        sum(
            1 for f in label_dir.iterdir()
            if f.suffix.lower() == ".txt"
        )
        if label_dir.exists()
        else 0
    )

    return img_count, lbl_count, label_dir


def check_annotation_quality(
    label_dir,
    class_names,
    split_name
):
    issues = []

    class_counts = {
        i: 0
        for i in range(len(class_names))
    }

    total_box_heights = []

    label_dir = Path(label_dir)

    if not label_dir.exists():
        return (
            [
                f"{split_name}: "
                f"label directory not found: {label_dir}"
            ],
            class_counts,
            total_box_heights
        )

    for lbl_file in sorted(label_dir.glob("*.txt")):
        lines = (
            lbl_file
            .read_text(encoding="utf-8")
            .strip()
            .splitlines()
        )

        if not lines:
            issues.append(
                f"EMPTY_LABEL [{split_name}]: "
                f"{lbl_file.name}"
            )
            continue

        seen_classes = set()

        for line in lines:
            parts = line.strip().split()

            # YOLO detection format:
            # class_id cx cy w h
            if len(parts) != 5:
                issues.append(
                    f"BAD_FORMAT [{split_name}]: "
                    f"{lbl_file.name} -> {line}"
                )
                continue

            try:
                cls = int(parts[0])
                cx, cy, w, h = map(
                    float,
                    parts[1:]
                )
            except ValueError:
                issues.append(
                    f"BAD_VALUE [{split_name}]: "
                    f"{lbl_file.name} -> {line}"
                )
                continue

            # class range check
            if not (0 <= cls < len(class_names)):
                issues.append(
                    f"BAD_CLASS [{split_name}]: "
                    f"{lbl_file.name} class={cls}"
                )
                continue

            # normalized coordinate range check
            for val, name in [
                (cx, "cx"),
                (cy, "cy"),
                (w, "w"),
                (h, "h")
            ]:
                if not (0.0 <= val <= 1.0):
                    issues.append(
                        f"OUT_OF_RANGE [{split_name}] "
                        f"{lbl_file.name} "
                        f"{name}={val:.4f}"
                    )

            # very small bbox height
            if h < 0.005:
                issues.append(
                    f"ZERO_HEIGHT [{split_name}] "
                    f"{lbl_file.name} "
                    f"h={h:.6f}"
                )

            # total class check
            if cls == 2:
                total_box_heights.append(
                    (h, lbl_file.name)
                )

                if h > 0.5:
                    issues.append(
                        f"TOTAL_FULL_RECEIPT "
                        f"[{split_name}] "
                        f"{lbl_file.name} "
                        f"w={w:.3f} "
                        f"h={h:.3f}"
                    )

            # duplicate class in one receipt
            if cls in seen_classes:
                issues.append(
                    f"DUPLICATE_CLASS "
                    f"[{split_name}] "
                    f"{lbl_file.name} "
                    f"class {cls}"
                )

            seen_classes.add(cls)

            class_counts[cls] += 1

    return (
        issues,
        class_counts,
        total_box_heights
    )


def check_image_resolutions(images_path_str):
    """
    Check the resolution of every image in the split.
    Returns Counter({(width, height): count}).
    """
    img_dir = Path(images_path_str)

    img_exts = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"
    }

    resolution_counts = Counter()

    if not img_dir.exists():
        return resolution_counts

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in img_exts:
            continue

        with PILImage.open(img_path) as image:
            resolution_counts[image.size] += 1

    return resolution_counts


# =========================================================
# Load Kaggle normalized data.yaml
# =========================================================

if "DATA_YAML_PATH" not in globals():
    raise NameError(
        "Run the previous cell first "
        "to create DATA_YAML_PATH."
    )


with DATA_YAML_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    data_config = yaml.safe_load(file)


required_keys = [
    "train",
    "val",
    "test",
    "names"
]

missing_keys = [
    key
    for key in required_keys
    if key not in data_config
]

if missing_keys:
    raise KeyError(
        f"data.yaml missing keys: "
        f"{missing_keys}"
    )


# =========================================================
# Class check
# =========================================================

class_names = normalize_class_names(
    data_config["names"]
)

class_count = int(
    data_config.get(
        "nc",
        len(class_names)
    )
)

if class_count != len(class_names):
    raise ValueError(
        f"Class mismatch: "
        f"nc={class_count}, "
        f"names={len(class_names)}"
    )


expected_class_names = [
    "date",
    "phone",
    "total"
]

if class_names == expected_class_names:
    print(
        "✓ Class order confirmed: "
        "date / phone / total"
    )
else:
    print(
        "⚠ WARNING: "
        f"Expected {expected_class_names}, "
        f"got {class_names}"
    )


# =========================================================
# Dataset split check
# =========================================================

print("\n=== Dataset split check ===")

all_issues = []

total_images = 0

all_class_counts = {
    i: 0
    for i in range(len(class_names))
}

all_total_heights = []


for split_name, split_path in [
    ("train", data_config["train"]),
    ("val", data_config["val"]),
    ("test", data_config["test"])
]:
    split_path = Path(split_path)

    if not split_path.exists():
        raise FileNotFoundError(
            f"{split_name} path missing: "
            f"{split_path}"
        )

    img_count, lbl_count, lbl_dir = (
        count_split_contents(
            split_path
        )
    )

    total_images += img_count

    status = (
        "✓"
        if img_count > 0 and lbl_count > 0
        else "✗"
    )

    print(
        f"\n[{status}] {split_name}: "
        f"{img_count} images, "
        f"{lbl_count} labels"
    )

    # -----------------------------------------------------
    # Full resolution check
    # -----------------------------------------------------

    resolution_counts = (
        check_image_resolutions(
            split_path
        )
    )

    print(
        f"{split_name} resolutions: "
        f"{dict(resolution_counts)}"
    )

    if (
        len(resolution_counts) == 1
        and (1024, 1024) in resolution_counts
        and resolution_counts[(1024, 1024)]
        == img_count
    ):
        print(
            f"✓ {split_name}: "
            f"all {img_count} images are "
            f"1024 × 1024"
        )
    else:
        print(
            f"⚠ {split_name}: "
            "image resolutions are not "
            "uniformly 1024 × 1024"
        )

    # -----------------------------------------------------
    # Annotation quality check
    # -----------------------------------------------------

    issues, counts, heights = (
        check_annotation_quality(
            lbl_dir,
            class_names,
            split_name
        )
    )

    all_issues.extend(
        issues
    )

    all_total_heights.extend(
        heights
    )

    for class_id, count in counts.items():
        all_class_counts[class_id] += count


print(
    f"\nTotal images: "
    f"{total_images}"
)


# =========================================================
# Class distribution
# =========================================================

print(
    "\n=== Class annotation counts ==="
)

for cls_id, cls_name in enumerate(
    class_names
):
    count = all_class_counts[cls_id]

    pct = (
        count / total_images * 100
        if total_images > 0
        else 0
    )

    print(
        f"class {cls_id} "
        f"({cls_name}): "
        f"{count}/{total_images} "
        f"({pct:.0f}%)"
    )


# =========================================================
# Total bbox check
# =========================================================

large_h = 0

if all_total_heights:
    heights = [
        h
        for h, _
        in all_total_heights
    ]

    large_h = sum(
        1
        for h in heights
        if h >= 0.5
    )

    print(
        "\n=== total bbox height "
        "distribution ==="
    )

    print(
        f"Count: {len(heights)}"
    )

    print(
        f"Min: {min(heights):.4f}, "
        f"Max: {max(heights):.4f}, "
        f"Avg: "
        f"{sum(heights) / len(heights):.4f}"
    )

    print(
        "Suspicious full receipt boxes: "
        f"{large_h}"
    )


# =========================================================
# Issue summary
# =========================================================

print(
    f"\n=== Annotation issues "
    f"({len(all_issues)}) ==="
)

if not all_issues:
    print(
        "✓ No annotation issues found."
    )
else:
    for issue in all_issues:
        print(issue)


if all_issues or large_h > 0:
    print(
        "\n⚠ Dataset issues detected."
    )
else:
    print(
        "\n✓ Dataset health check passed."
    )


# =========================================================
# Final dataset summary
# =========================================================

print(
    "\n=== Dataset configuration ==="
)

print(
    f"train: "
    f"{data_config['train']}"
)

print(
    f"val:   "
    f"{data_config['val']}"
)

print(
    f"test:  "
    f"{data_config['test']}"
)

print(
    f"nc: {class_count}, "
    f"names: {class_names}"
)

In [ ]:
# 手順5: YOLO11n を 1024px 設定で学習する

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=DATA_YAML_PATH,

    # 基本学習設定
    epochs=300,
    imgsz=1024,
    batch=8,
    device=0,
    patience=80,

    # レシート向けの軽いデータ拡張
    mosaic=0.2,
    mixup=0.0,
    cutmix=0.0,

    degrees=0.0,
    translate=0.05,
    scale=0.2,
    shear=0.0,
    perspective=0.0,

    # レシートの向きを維持するため反転しない
    fliplr=0.0,
    flipud=0.0,

    # 再現性を維持する
    deterministic=True,
    seed=42,

    # 学習結果の保存先
    project="/kaggle/working/runs/detect",
    name="receipt_yolo11n_1024",
    exist_ok=False,

    # 検証結果を保存する
    val=True,
    plots=True,
)

## Validation Threshold Scan

In [ ]:
test_results = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=1024,
    project=str(EVAL_DIR),
    name="test_benchmark_1024",
    exist_ok=True,
    plots=True,
    verbose=False,
)

In [ ]:
# Helper: Ultralytics の評価指標を安全に取得する

def read_box_metric(metrics, attr_name, result_key):
    box_metrics = getattr(metrics, "box", None)

    if box_metrics is not None and hasattr(box_metrics, attr_name):
        value = getattr(box_metrics, attr_name)

        if value is not None:
            return float(value)

    results_dict = getattr(metrics, "results_dict", {})

    if isinstance(results_dict, dict) and result_key in results_dict:
        return float(results_dict[result_key])

    return float("nan")

In [ ]:
# 手順7: best.pt を test split で正式評価する

from ultralytics import YOLO

best_model = YOLO(str(BEST_PT_PATH))

print(f"Model: {BEST_PT_PATH}")
print("Split: test")
print(f"Image size: {IMG_SIZE}")
print("Confidence: Ultralytics validation default")

test_results = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=IMG_SIZE,

    # benchmark 評価では deployment threshold を固定しない
    project=str(EVAL_DIR),
    name="test_benchmark_1024",

    plots=True,
    verbose=False,
)

precision = read_box_metric(
    test_results,
    "mp",
    "metrics/precision(B)"
)

recall = read_box_metric(
    test_results,
    "mr",
    "metrics/recall(B)"
)

map50 = read_box_metric(
    test_results,
    "map50",
    "metrics/mAP50(B)"
)

map50_95 = read_box_metric(
    test_results,
    "map",
    "metrics/mAP50-95(B)"
)

print("\nFinal test benchmark metrics:")
print(f" - Precision:   {precision:.4f}")
print(f" - Recall:      {recall:.4f}")
print(f" - mAP50:       {map50:.4f}")
print(f" - mAP50-95:    {map50_95:.4f}")

In [19]:
test_results = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=1024,
    project=str(EVAL_DIR),
    name="test_benchmark_1024",
    exist_ok=True,
    plots=True,
    verbose=False,
)

Ultralytics 8.4.141 🚀 Python-3.12.13 torch-2.6.0+cu126 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 2.0±0.8 ms, read: 131.9±20.9 MB/s, size: 80.1 KB)
val: Scanning /kaggle/input/datasets/lunarinaleeluna/receipt-detection-system-v4i-260906-yolov11/receipt-detection-system.v4i.yolov11/test/labels... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 778.9it/s 0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/lunarinaleeluna/receipt-detection-system-v4i-260906-yolov11/receipt-detection-system.v4i.yolov11/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.0it/s 0.5s
                   all         13         38      0.941       0.95      0.969      0.624
Speed: 1.2ms preprocess, 13.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/eval_c

In [20]:
# 手順8: 1024px 学習成果物をZIPにまとめる (Kaggle版)

from pathlib import Path
import json
import zipfile
import hashlib


# =========================================================
# Training output paths
# =========================================================

RUN_NAME = "receipt_yolo11n_1024"

TRAIN_DIR = Path(
    "/kaggle/working/runs/detect/receipt_yolo11n_1024"
)

WEIGHTS_DIR = TRAIN_DIR / "weights"

BEST_PT_PATH = WEIGHTS_DIR / "best.pt"
LAST_PT_PATH = WEIGHTS_DIR / "last.pt"


if not BEST_PT_PATH.exists():
    raise FileNotFoundError(
        f"best.pt not found: {BEST_PT_PATH}"
    )


# =========================================================
# Save model SHA256
# =========================================================

best_pt_sha256 = hashlib.sha256(
    BEST_PT_PATH.read_bytes()
).hexdigest()

sha256_path = TRAIN_DIR / "best_pt_sha256.txt"

sha256_path.write_text(
    best_pt_sha256,
    encoding="utf-8"
)

print("best.pt SHA256:")
print(best_pt_sha256)


# =========================================================
# Save training summary
# =========================================================

training_summary = {
    "run_name": RUN_NAME,
    "model": "yolo11n",
    "imgsz": 1024,
    "epochs_requested": 300,
    "epochs_completed": 245,
    "best_epoch": 165,
    "batch": 8,
    "patience": 80,
    "seed": 42,
    "classes": [
        "date",
        "phone",
        "total"
    ],
    "validation_metrics": {
        "precision": 0.993,
        "recall": 1.000,
        "mAP50": 0.995,
        "mAP50-95": 0.731
    }
}

training_summary_path = (
    TRAIN_DIR / "training_summary.json"
)

with training_summary_path.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        training_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Training summary saved:",
    training_summary_path
)


# =========================================================
# Artifact candidates
# =========================================================

artifact_candidates = [
    BEST_PT_PATH,
    LAST_PT_PATH,

    TRAIN_DIR / "results.png",
    TRAIN_DIR / "PR_curve.png",
    TRAIN_DIR / "P_curve.png",
    TRAIN_DIR / "R_curve.png",
    TRAIN_DIR / "F1_curve.png",

    TRAIN_DIR / "args.yaml",
    TRAIN_DIR / "results.csv",

    training_summary_path,
    sha256_path,
]


# Add confusion matrix files
confusion_matrix_files = list(
    TRAIN_DIR.glob(
        "confusion_matrix*.png"
    )
)

artifact_candidates.extend(
    confusion_matrix_files
)


# =========================================================
# Create ZIP
# =========================================================

ARTIFACT_ZIP_PATH = (
    TRAIN_DIR /
    "yolo11n_1024_training_artifacts.zip"
)

if ARTIFACT_ZIP_PATH.exists():
    ARTIFACT_ZIP_PATH.unlink()


with zipfile.ZipFile(
    ARTIFACT_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    seen_names = set()

    for artifact_path in artifact_candidates:

        if artifact_path.exists():

            arcname = artifact_path.name

            if arcname in seen_names:
                arcname = (
                    f"{artifact_path.parent.name}_"
                    f"{arcname}"
                )

            seen_names.add(arcname)

            archive.write(
                artifact_path,
                arcname=arcname
            )

            print(
                f"Added to ZIP: {arcname}"
            )

        else:
            print(
                "Skipped missing artifact:",
                artifact_path.name
            )


print("")
print(
    "Training results directory:",
    TRAIN_DIR
)

print(
    "Weights directory:",
    WEIGHTS_DIR
)

print(
    "Artifact ZIP created at:",
    ARTIFACT_ZIP_PATH
)

best.pt SHA256:
a450a8d56e6230e1e3ee598906a07815ea3b847f986d6af17a6c036629ed6ca4
Training summary saved: /kaggle/working/runs/detect/receipt_yolo11n_1024/training_summary.json
Added to ZIP: best.pt
Added to ZIP: last.pt
Added to ZIP: results.png
Skipped missing artifact: PR_curve.png
Skipped missing artifact: P_curve.png
Skipped missing artifact: R_curve.png
Skipped missing artifact: F1_curve.png
Added to ZIP: args.yaml
Added to ZIP: results.csv
Added to ZIP: training_summary.json
Added to ZIP: best_pt_sha256.txt
Added to ZIP: confusion_matrix_normalized.png
Added to ZIP: confusion_matrix.png

Training results directory: /kaggle/working/runs/detect/receipt_yolo11n_1024
Weights directory: /kaggle/working/runs/detect/receipt_yolo11n_1024/weights
Artifact ZIP created at: /kaggle/working/runs/detect/receipt_yolo11n_1024/yolo11n_1024_training_artifacts.zip


In [22]:
# 手順9: Kaggle で 1024px 学習成果物をダウンロードできるように表示する

from IPython.display import display, FileLink


print("=== YOLO11n 1024 Training Artifacts ===")


# =========================================================
# best.pt
# =========================================================

print("\nbest.pt:")

if BEST_PT_PATH.exists():
    display(
        FileLink(str(BEST_PT_PATH))
    )

    print(
        f"Size: "
        f"{BEST_PT_PATH.stat().st_size / 1024 / 1024:.2f} MB"
    )

else:
    print(
        f"ERROR: best.pt not found: "
        f"{BEST_PT_PATH}"
    )


# =========================================================
# Training artifacts ZIP
# =========================================================

print("\nyolo11n_1024_training_artifacts.zip:")

if ARTIFACT_ZIP_PATH.exists():
    display(
        FileLink(str(ARTIFACT_ZIP_PATH))
    )

    print(
        f"Size: "
        f"{ARTIFACT_ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MB"
    )

else:
    print(
        f"ERROR: artifacts ZIP not found: "
        f"{ARTIFACT_ZIP_PATH}"
    )


# =========================================================
# Final summary
# =========================================================

print("\n=== Artifact Paths ===")

print(
    f"best.pt:\n"
    f"{BEST_PT_PATH}"
)

print(
    f"\ntraining artifacts ZIP:\n"
    f"{ARTIFACT_ZIP_PATH}"
)

=== YOLO11n 1024 Training Artifacts ===

best.pt:


/kaggle/working/runs/detect/receipt_yolo11n_1024/weights/best.pt

Size: 5.32 MB

yolo11n_1024_training_artifacts.zip:


/kaggle/working/runs/detect/receipt_yolo11n_1024/yolo11n_1024_training_artifacts.zip

Size: 9.80 MB

=== Artifact Paths ===
best.pt:
/kaggle/working/runs/detect/receipt_yolo11n_1024/weights/best.pt

training artifacts ZIP:
/kaggle/working/runs/detect/receipt_yolo11n_1024/yolo11n_1024_training_artifacts.zip


In [30]:
import shutil
from pathlib import Path

folder = Path("/kaggle/working/weights")

if folder.exists():
    shutil.rmtree(folder)
    print("Deleted:", folder)
else:
    print("Folder not found:", folder)

Deleted: /kaggle/working/weights
